# 1 Deploy a Model with vLLM on Klique

**Purpose:**
Turn a model name into a working, authenticated endpoint, using Klique's vLLM Model Deployment app.

**What we're doing**

We launch the vLLM Model Deployment app with one small model, wait for its endpoint to become active, and then send it a request from this notebook.

**Why we're doing it**

* Everything later in the lab, including the LLM Gateway in Section 2, needs a running endpoint
* To see which parts Klique handles for you (GPU scheduling, the vLLM server, the secure route) and which parts you decide (the model, the queue, the endpoint name)
* To learn the difference between your Klique login and the token that calls a model

**What to look for**

* How few fields you have to change on the launch form
* The instance moving from **Loading** to **Active**
* The same endpoint appearing in the instance dashboard and in Model Endpoints

**Output**

A running model endpoint, and a `.env` file at the project root holding its URL, its model name, and a gateway token.


## 1.1 How Klique Serves a Model

You do not install or run vLLM yourself. Klique ships it as an application, **vLLM Model Deployment**. Each time you launch the app, Klique creates an **instance**: a task that runs on the queue you pick, starts a vLLM server on that queue's GPU, and publishes the server through the **AI Application Gateway** as a **model endpoint**.

An instance moves through four states. You see them on the instance dashboard:

| State | What is happening |
|---|---|
| **Loading** | The instance is waiting for an agent on the queue, pulling the image, or downloading and loading the model weights |
| **Active** | The vLLM server is up and the endpoint answers requests |
| **Idle** | No requests have arrived recently. The endpoint still answers. |
| **Stopped** | You stopped it, it reached its idle time limit, or something failed |

> **Note:** A stopped instance releases its GPU. A running one holds it, even when idle. Section 1.9 shows how to stop yours.


## 1.2 Choose a Model

The launch form's **Model** field takes either:

- A **HuggingFace model name**, such as `Qwen/Qwen2.5-1.5B-Instruct`, which Klique downloads when the instance starts, or
- A **Klique Model ID**, for a model already registered in Klique

This lab uses **`Qwen/Qwen2.5-1.5B-Instruct`**. It is small enough to load quickly on a single GPU, and it is *ungated*, so you do not need a HuggingFace token.

> **Tip:** If you pick a gated model instead, such as a Llama model, paste a HuggingFace token into the form's **HuggingFace Token** field. Without it, the instance stays in Loading because it cannot download the weights.


## 1.3 Launch the vLLM Model Deployment App

Log in to the Klique WebApp and open **Applications**. Select **vLLM Model Deployment**, then click **Launch New** (the plus icon) to open the launch form.

<!-- screenshot: images/applications.png (Applications page with vLLM Model Deployment selected) -->

Fill in these fields and leave everything else at its default:

| Field | Value | Why it matters |
|---|---|---|
| **Instance name** | `qwen-lab` | Shown in the instance list |
| **Project** | Your project | Controls who can see the instance |
| **Compute Resource** | Your GPU queue | The queue decides which machine and GPU the instance runs on. It needs an assigned agent. |
| **AI Gateway Route** | *Leave empty* | Klique creates an ephemeral endpoint for this instance |
| **Model** (under *Deployed Models*) | `Qwen/Qwen2.5-1.5B-Instruct` | What vLLM serves |
| **Model Endpoint Name** | `qwen-lab` | The name clients send as `"model"` in every request. You will need it in 1.6. |

<!-- screenshot: images/launch_form.png (launch form with the fields above filled in) -->

Open **Advanced** and look at **Idle Time Limit (Hours)**. It stops the instance after that many hours without traffic, which frees the GPU if you forget. Set it to `2` for this lab.

Click **Launch**.

> **Note:** The rest of the form, from GPU memory utilization to tool-call parsers, maps to vLLM's own settings. The defaults are right for a small model on one GPU. See the [vLLM Model Deployment reference](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment) when you need them.

> **Tip:** To repeat this launch later without retyping, open the instance's menu (⋮) and choose **Export Configuration**. **Import Configuration** in the instance list menu opens the launch form prefilled from that file.


## 1.4 Watch the Instance Come Up

Click the new instance in the instance list to open its dashboard. The status indicator starts at **Loading**. Open the **console log** on the dashboard to watch the progress: the image starts, the model weights download, and vLLM loads them onto the GPU.

<!-- screenshot: images/dashboard_loading.png (instance dashboard, Loading, console log visible) -->

When loading finishes, the status changes to **Active** and the **Deployed models** table shows your model and its **Endpoint** URL.

<!-- screenshot: images/dashboard_active.png (instance dashboard, Active, Deployed models table with Endpoint) -->

Now open **Model Endpoints** from the WebApp menu. Your endpoint appears in the **Loading** tab first, then moves to the **Active** tab with its URL.

<!-- screenshot: images/model_endpoints.png (Model Endpoints, Active tab, your endpoint listed) -->

> **Checkpoint:** your endpoint is in **Model Endpoints → Active**. Copy its URL; you need it in 1.6.

If the instance stays in Loading or stops, check the console log and the Troubleshooting table at the end of this notebook.


## 1.5 Generate a Gateway Token

The endpoint does not accept your Klique login. Every request needs an **AI Application Gateway token**, which the gateway checks before the request reaches vLLM.

Open **Settings**. In the **Workspace** section, find **AI Application Gateway Tokens** and click **Generate a Token**. Enter a **Label** such as `klique-lab`, an **Expiration** in days, and click **Generate**. The token is copied to your clipboard.

<!-- screenshot: images/generate_token.png (Generate a Token dialog) -->

> **Tip:** Paste the token somewhere safe right away; you need it in the next step. To revoke it later, hover over its row in the token table and click the trash icon.

> **Note:** Your Klique login and API credentials manage instances. The gateway token only calls models. Keeping them separate means a leaked model token cannot launch or stop anything.

## 1.6 Create Your `.env` File

The URL, the model name, and the token go into a `.env` file at the **root of the project**, not in this folder. Every notebook in the lab reads the same file through `config.py`, so you set these values once.

| Variable | Description |
|---|---|
| `KLIQUE_BASE_URL` | The endpoint URL from 1.4, ending in `/v1` |
| `KLIQUE_MODEL` | The **Model Endpoint Name** from 1.3 (`qwen-lab`) |
| `KLIQUE_GATEWAY_TOKEN` | The token from 1.5 |

Run the cell below. It asks for each value. The token prompt hides what you type, so the token never appears in the notebook. If `.env` already exists, the cell leaves it alone.

In [ ]:
from getpass import getpass
from pathlib import Path

env_path = Path("..") / ".env"

if env_path.exists():
    print(f"{env_path.resolve()} already exists. Edit it in a text editor to change values.")
else:
    base_url = input("Endpoint URL: ").strip().rstrip("/")
    if not base_url.endswith("/v1"):
        base_url += "/v1"
    model = input("Model Endpoint Name: ").strip()
    token = getpass("AI Application Gateway token: ").strip()

    env_path.write_text(
        f"KLIQUE_BASE_URL={base_url}\n"
        f"KLIQUE_MODEL={model}\n"
        f"KLIQUE_GATEWAY_TOKEN={token}\n"
    )
    print(f"Wrote {env_path.resolve()}")

If you prefer, create the file in a text editor instead. Put it in the project root, next to `config.py`:

```
KLIQUE_BASE_URL=https://<your endpoint>/v1
KLIQUE_MODEL=qwen-lab
KLIQUE_GATEWAY_TOKEN=<your token>
```

> **Note:** `.env` is listed in `.gitignore`, so it is never committed. Never paste a token into a notebook cell: the cell's text is saved in the `.ipynb` file and in every copy you share.


## 1.7 Load Values from `.env`

In [ ]:
import sys
sys.path.insert(0, "..")
from config import KLIQUE_BASE_URL as base_url, KLIQUE_MODEL as model, KLIQUE_GATEWAY_TOKEN as token
import requests

url_models = f"{base_url}/models"
url_chat = f"{base_url}/chat/completions"

print(f"Endpoint: {base_url}")
print(f"Model:    {model}")
print(f"Token:    {token[:8]}...")

Notice that the token is intentionally truncated in the output above. Anyone who can see a full token in an output, a screenshot, or a recorded demo has a working credential. The first few characters are enough to confirm the right token loaded.


## 1.8 Test the Endpoint

### 1.8.1 List the Models the Endpoint Serves

vLLM serves an OpenAI-compatible API, so the same calls work here that work against any OpenAI-style service. Start by asking the endpoint which models it serves. The token goes in the `Authorization` header as a bearer token.

In [ ]:
headers = {"Authorization": f"Bearer {token}"}

response = requests.get(url_models, headers=headers, timeout=60)
if response.ok:
    print("Models served by this endpoint:")
    for m in response.json()["data"]:
        print(f"\t- {m['id']}")
else:
    print(f"❌ HTTP {response.status_code}: {response.text[:300]}")

The list should contain your **Model Endpoint Name**, `qwen-lab`. If it does not, `KLIQUE_MODEL` in `.env` will not match and the next call fails.

### 1.8.2 Send a Chat Completion

In [ ]:
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}
data = {
    "model": model,
    "messages": [{"role": "user", "content": "In one sentence, what is Klique?"}],
    "max_tokens": 128,
}

response = requests.post(url_chat, headers=headers, json=data, timeout=60)
if response.ok:
    print("✅ Response:")
    print(response.json()["choices"][0]["message"]["content"])
    print(f"\nTokens used: {response.json()['usage']['total_tokens']}")
else:
    print(f"❌ HTTP {response.status_code}: {response.text[:300]}")

You will get *an* answer, and it may well be wrong: a 1.5B model has probably never heard of Klique. That's fine. The point of this cell is that the request reached your instance and came back.

> **Checkpoint:** the cell prints `✅ Response:` followed by text. Your model is deployed on Klique and answering through the AI Application Gateway.

Go back to the instance dashboard. The **Total number of requests**, **Tokens per second**, and **Latency** panels now show your calls.

<!-- screenshot: images/dashboard_metrics.png (dashboard request and latency panels after the test calls) -->


## 1.9 Stop the Instance When You Are Done

**If you are continuing to Section 2, leave the instance running.** Section 2 uses this endpoint.

When you are finished, stop it to release the GPU. In the instance list, open the instance's menu (⋮) and choose **Stop**. The endpoint leaves the **Active** tab in Model Endpoints and stops answering.

| Action | What it does |
|---|---|
| **Stop** | Releases the GPU and the endpoint. The instance and its configuration stay, so you can relaunch it. |
| **Relaunch** | Starts a fresh instance with the same configuration and removes the original |
| **Delete** | Removes the instance |

If you forget, the **Idle Time Limit** you set in 1.3 stops it for you.


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| Stays in **Loading** | No agent is assigned to the queue you picked, or the model is gated and no HuggingFace token was set. Check the console log. |
| Goes to **Stopped** while loading | The model does not fit the GPU (look for an out-of-memory error in the console log), or the vLLM version does not support the model. Try a smaller model. |
| `401` from the endpoint | The gateway token is wrong, expired, or revoked, or you used something other than a gateway token. Generate a new one in 1.5. |
| `404` from the endpoint | `KLIQUE_BASE_URL` is wrong. Compare it with the URL in Model Endpoints, and check that it ends in `/v1`. |
| The model name is rejected | `KLIQUE_MODEL` does not match the **Model Endpoint Name** from the launch form. |
| Connection error or timeout | This machine cannot reach the endpoint's network, or idle shutdown has already stopped the instance. |


## Want to Go Further?

`vllm_klique_test_clean.ipynb` in this folder goes beyond a single request: raw completions, system prompts, and handling the `<think>` block that reasoning models such as Qwen3 return.

## Next: Putting the Endpoint Behind the LLM Gateway

You now have a model running on Klique and answering requests. But look at what every client needed to use it: the raw endpoint URL, the exact model name, and its own gateway token. Nothing decides *who* may call this model, and nothing tracks what they use.

That is the job of the [LLM Gateway](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway), Klique's central point of control for LLM traffic. It is deployed as part of the AI Application Gateway, and it adds:

- **Governed access:** profiles decide which users, groups, and projects can reach which models
- **One route for clients:** applications call the gateway instead of each model's endpoint
- **Cost control:** token usage is metered per user and per model, with budgets and rate limits
- **Visibility:** usage appears in the AI Usage dashboard

**Next:** In Section 2 we put the endpoint you just deployed behind the LLM Gateway and call it the governed way.